## Imports  

#### Loads the libraries we use: json to read the data, re and string for cleaning text, Counter for counting words, BeautifulSoup to remove HTML, sklearn for the classifiers and nltk for bigrams.

In [3]:
import pandas as pd
import json
import re
import string
import math
import numpy as np
from collections import Counter, defaultdict
from bs4 import BeautifulSoup
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from nltk import bigrams

## Load Data

#### The data is in JSONL format, so every line is one JSON object (one text). load_jsonl reads the file line by line and puts each text into a list. We load train.jsonl and dev.jsonl.

In [4]:
def load_jsonl(path):
    data = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            record = json.loads(line)
            data.append(record)
    return data

train = load_jsonl('train.jsonl')
dev = load_jsonl('dev.jsonl')

## Explore Data

#### We look at the data before doing anything. Each text has an id, text, model, label and genre. Label 0 = human, label 1 = AI.
- Train has 23,707 texts and dev has 1,795.
- About 62% of train is AI, so the data is imbalanced.
- Fiction is about 50/50, but essays and news are about 80% AI.
- Human texts are longer on average (695 words vs 575) and vary more in length.

In [5]:
print(len(train), len(dev))
print(train[0].keys())
print(train[0])
print(Counter(r['label'] for r in train))
print(Counter(r['label'] for r in dev))
print(Counter((r['label'], r['model']) for r in train))
print(Counter(r['genre'] for r in train))
print(Counter((r['label'], r['genre']) for r in train))
for label in set(r['label'] for r in train):
    lengths = [len(r['text'].split()) for r in train if r['label'] == label]
    print(label, 'min:', min(lengths), 'max:', max(lengths), 'gns:', round(sum(lengths)/len(lengths)))

23707 1795
dict_keys(['id', 'text', 'model', 'label', 'genre'])
{'id': 'ea468d03-1973-5039-86b2-ff225bb92c4e', 'text': 'Duke Ellington, a titan of jazz, revolutionized the genre through his innovative compositions, showcasing a remarkable ability to integrate voice and instrumental music. Among the notable figures who contributed to this artistic symphony was Ivie Anderson, whose scat singing mirrored the improvisational prowess of musicians like Nanton. In "Ring Dem Bells," Ellington ingeniously interweaves scat singing as a dialogue with saxophones, creating a dynamic call and response that underscores his vision of music as a fluid conversation between voices and instruments.\n\nEllington consistently emphasized the voice as an instrument of equal importance to traditional brass and woodwinds, orchestrating his compositions with a keen ear for vocal qualities. This approach is vividly demonstrated in "Mood Indigo," where his orchestration skills shine through non-traditional chord a

## Pre-processing

#### clean_text normalizes the text: lowercase, removes HTML, URLs, numbers and punctuation, and removes extra spaces. tokenize splits the cleaned text into words on spaces. We save the tokens for every text in r['tokens'] so we don't have to clean again later.

In [6]:
def clean_text(text):
    text = text.lower()
    text = BeautifulSoup(text, "html.parser").get_text()
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'\d+', '', text)
    text = text.translate(str.maketrans('', '', string.punctuation))
    text = re.sub(r'\W+', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

In [7]:
print(train[0]['text'][:300])
print()
print(clean_text(train[0]['text'])[:300])

Duke Ellington, a titan of jazz, revolutionized the genre through his innovative compositions, showcasing a remarkable ability to integrate voice and instrumental music. Among the notable figures who contributed to this artistic symphony was Ivie Anderson, whose scat singing mirrored the improvisati

duke ellington a titan of jazz revolutionized the genre through his innovative compositions showcasing a remarkable ability to integrate voice and instrumental music among the notable figures who contributed to this artistic symphony was ivie anderson whose scat singing mirrored the improvisational 


In [8]:
def tokenize(text):
    return clean_text(text).split()

print(tokenize(train[0]['text'])[:30])

['duke', 'ellington', 'a', 'titan', 'of', 'jazz', 'revolutionized', 'the', 'genre', 'through', 'his', 'innovative', 'compositions', 'showcasing', 'a', 'remarkable', 'ability', 'to', 'integrate', 'voice', 'and', 'instrumental', 'music', 'among', 'the', 'notable', 'figures', 'who', 'contributed', 'to']


In [9]:
for r in train:
    r['tokens'] = tokenize(r['text'])
for r in dev:
    r['tokens'] = tokenize(r['text'])

print(train[0]['tokens'][:30])

['duke', 'ellington', 'a', 'titan', 'of', 'jazz', 'revolutionized', 'the', 'genre', 'through', 'his', 'innovative', 'compositions', 'showcasing', 'a', 'remarkable', 'ability', 'to', 'integrate', 'voice', 'and', 'instrumental', 'music', 'among', 'the', 'notable', 'figures', 'who', 'contributed', 'to']


## Vocabulary

#### We count words separately for human and AI texts with Counter. Humans have 112,763 unique words and AI has 79,963, so humans use a more varied vocabulary. The most common words are function words like "the", "and" and "of".

We also found 281 texts with Chinese characters, all written by AI (mostly deepseek). A model could learn "Chinese = AI" as a shortcut.

72.5% of the unique words in train never show up in dev. Most of them are rare words like names and typos.

In [10]:
human_Vocab = Counter()
machine_Vocab = Counter()

for r in train:
    if r['label'] == 0:
        human_Vocab.update(r['tokens'])
    else:
        machine_Vocab.update(r['tokens'])

print('Human:', len(human_Vocab), 'unique ord')
print('Machine:', len(machine_Vocab), 'unique ord')

Human: 112763 unique ord
Machine: 79963 unique ord


In [11]:
print(human_Vocab.most_common(30))
print()
print(machine_Vocab.most_common(30))

[('the', 357452), ('and', 198040), ('of', 184763), ('to', 173344), ('a', 152700), ('in', 109274), ('was', 83214), ('i', 81013), ('that', 78351), ('he', 78187), ('it', 68253), ('his', 63385), ('with', 52809), ('for', 51641), ('her', 51619), ('as', 51072), ('you', 50897), ('had', 48441), ('is', 43679), ('she', 41618), ('not', 38855), ('on', 38624), ('at', 38526), ('but', 37344), ('be', 34163), ('have', 30589), ('said', 27227), ('this', 26350), ('by', 25942), ('him', 25561)]

[('the', 575528), ('of', 325045), ('a', 260664), ('and', 218424), ('to', 195536), ('in', 141319), ('that', 100297), ('his', 99321), ('with', 95390), ('as', 81878), ('her', 77276), ('for', 64847), ('he', 56215), ('was', 54932), ('i', 49502), ('it', 45651), ('by', 41466), ('their', 41313), ('on', 40505), ('is', 40059), ('had', 36865), ('not', 35497), ('this', 34894), ('but', 34169), ('she', 33571), ('from', 31793), ('an', 29253), ('its', 26919), ('at', 26438), ('my', 24816)]


In [12]:
all_Vocab = human_Vocab + machine_Vocab
print(sorted(all_Vocab)[:50])
print(sorted(all_Vocab)[-50:])

['a', 'aa', 'aaa', 'aaaaaaaaaaaahahahahahahha', 'aaahouch', 'aaan', 'aaarh', 'aablest', 'aachen', 'aaeven', 'aahas', 'aahing', 'aaisnt', 'aal', 'aalamat', 'aan', 'aap', 'aappendix', 'aaqii', 'aaraaf', 'aarhus', 'aaron', 'aaronever', 'aarons', 'aashe', 'aasheim', 'aasm', 'aautoloadfächerous', 'ab', 'aba', 'aback', 'abacterial', 'abacus', 'abaft', 'abandon', 'abandoned', 'abandoneda', 'abandonedof', 'abandonedthe', 'abandonedwhat', 'abandoning', 'abandonment', 'abandons', 'abapical', 'abase', 'abasement', 'abashd', 'abashed', 'abashedly', 'abasin']
['这可不像小事', '这是他熟悉的土地', '这样的行为不会被容忍', '这还没完', '选择坚守我们的岗位', '遮蔽了天空', '那一刻', '那些为了鸡毛蒜皮的小事而大动肝火的日子', '那些人仿佛都被时间带走了', '那些建筑虽然依旧矗立', '那些无谓的争执', '那些还记得', '那些陡峭的峡湾和高耸的山脉仿佛触手可及', '那双深邃的眼睛透着岁月的沧桑与智慧的光芒', '那或许是曾经的某个熟人', '那时的他们', '那时的他们常常在月光下讲述自己的故事', '那时的他还年轻', '那时的海面被月光覆盖', '那是他在海上多年的伴侣', '那男子大声问道', '那艘船', '那艘船早已被人遗忘', '那里却矗立着一座新的高楼', '那里如今已经换上了新的招牌', '那里如今摆满了他从未见过的新鲜海鲜', '那里或许还残留着一些他记忆中的影子', '那里曾经是他和同伴们常常聚会的地方', '都似乎在这一次的摔打之后烟消云散了', '银色的头发在午后斜阳中泛着微光', 

In [13]:
Chinese = [r for r in train if re.search(r'[\u4e00-\u9fff]', r['text'])]
print(len(Chinese))
print(Counter((r['label'], r['model']) for r in Chinese))

281
Counter({(1, 'deepseek-r1-distill-qwen-32b'): 215, (1, 'ministral-8b-instruct-2410'): 39, (1, 'llama-3.1-8b-instruct'): 10, (1, 'qwen1.5-72b-chat-8bit'): 5, (1, 'gemini-1.5-pro'): 3, (1, 'text-bison-002'): 2, (1, 'gpt-4o-mini'): 2, (1, 'o3-mini'): 1, (1, 'gemini-2.0-flash'): 1, (1, 'gemini-pro-paraphrase'): 1, (0, 'human'): 1, (1, 'gemini-pro'): 1})


In [14]:
train_vocab = set(t for r in train for t in r['tokens'])
dev_vocab = set(t for r in dev for t in r['tokens'])

only_train = train_vocab - dev_vocab
print(round(len(only_train) / len(train_vocab) * 100, 1), '%')

72.5 %


## Question 4

**a) What kinds of words do you see?**
The top of the list is almost only function words like the, and, of, to. Humans use "i", "was" and "he" more, so they write more like a story in first person and past tense. The AI uses "of", "a", "with" and "as" more. In the alphabetical list there are also expressions like "aaaaahahaha" and "aarh", and at the end there is Chinese text.

**b) Does the text normalization work well?**
Mostly. Capital letters and punctuation are gone. But because punctuation is deleted instead of replaced with a space, some words get merged, like "aaeven". Special characters like ’ and – are not removed by string.punctuation, but the \W+ line after catches them and turns them into a space, so "don’t" becomes "don t". And Chinese is not split into words, because there are no spaces, so whole sentences become one token.

**c) Are there marked differences between human_Vocab and machine_Vocab?**
Yes. Humans have 112,763 unique words and the AI only 79,963, even though there are fewer human texts. The AI texts also have more tokens in total (about 14,606 × 575 ≈ 8.4M vs 9,101 × 695 ≈ 6.3M), so the AI writes more text but still uses fewer unique words. So humans have a more varied vocabulary.

**d) What % of tokens were seen in train but not dev?**
72.5%. Most of them are rare words like names and typos, and dev is much smaller than train.

**e) Other observations**
- The dataset is imbalanced: about 62% AI and 38% human.
- Fiction is about 50/50, but essays and news are about 80% AI.
- Human texts are longer on average and vary more in length.
- 281 texts contain Chinese, and they are all written by AI (mostly deepseek). For example line 272 in train.jsonl, where the model switches to Chinese in the middle of an English sentence. A model could learn "Chinese = AI" as a shortcut.

## Baseline

#### Before training a model we check what accuracy we get by always guessing AI. This gives 0.644, because 64% of dev is AI. A model has to beat this to be useful. Because the data is imbalanced, we also look at F1 and not only accuracy.

In [20]:
train_texts = [' '.join(r['tokens']) for r in train]
dev_texts = [' '.join(r['tokens']) for r in dev]
y_train = [r['label'] for r in train]
y_dev = [r['label'] for r in dev]

baseline = Counter(y_dev)[1] / len(y_dev)
print('Baseline accuracy (always guess AI):', round(baseline, 3))

Baseline accuracy (always guess AI): 0.644


## Bag of words + Logistic Regression

#### CountVectorizer turns each text into a vector where each column is a word and the value is how many times it appears (bag of words). We fit it only on train and use transform on dev, so words that only appear in dev are ignored. Otherwise it would be data contamination.

We changed token_pattern so words with one letter like "i" and "a" are kept, because they were different between humans and AI in lesson 1.

Logistic regression learns a weight for each word. Result: accuracy 0.987 and F1 0.98 for human and 0.99 for AI. Only 14 human texts were guessed as AI and 9 AI texts as human.

In [16]:
vectorizer = CountVectorizer(token_pattern=r"(?u)\b\w+\b")
X_train = vectorizer.fit_transform(train_texts)
X_dev = vectorizer.transform(dev_texts)
print('Shape:', X_train.shape)

model = LogisticRegression(max_iter=2000)
model.fit(X_train, y_train)
y_pred = model.predict(X_dev)

print('Accuracy:', round(accuracy_score(y_dev, y_pred), 3))
print('Confusion matrix:\n', confusion_matrix(y_dev, y_pred))
print(classification_report(y_dev, y_pred, target_names=['human', 'AI']))

Shape: (23707, 139096)
Accuracy: 0.987
Confusion matrix:
 [[ 625   14]
 [   9 1147]]
              precision    recall  f1-score   support

       human       0.99      0.98      0.98       639
          AI       0.99      0.99      0.99      1156

    accuracy                           0.99      1795
   macro avg       0.99      0.99      0.99      1795
weighted avg       0.99      0.99      0.99      1795



#### We look at the weights the model learned. Words that point to AI are formal words like "significant", "despite", "crucial" and "however". Words that point to human are more like normal storytelling, like "because", "cried" and "wednesday".

In [17]:
words = vectorizer.get_feature_names_out()
coefs = model.coef_[0]

print('Most AI:', [words[i] for i in np.argsort(coefs)[-20:][::-1]])
print('Most human:', [words[i] for i in np.argsort(coefs)[:20]])

Most AI: ['significant', 'despite', 'crucial', 'however', 'concerns', 'challenges', 'particularly', 'investigation', 'individuals', 'remains', 'approach', 'confess', 'amidst', 'stated', 'emphasized', 'phebe', 'further', 'grandfather', 'expressed', 'incident']
Most human: ['because', 'cried', 'other', 'wednesday', 'although', 'two', 'fact', 'answered', 'says', 'few', 'month', 'gave', 'wife', 'having', 'does', 'wrote', 'scarcely', 'tuesday', 'order', 'pretty']


## TF-IDF

#### Same setup, but the vectors are TF-IDF instead of counts. TF-IDF gives low weight to words that are in almost every text and high weight to rare words. It also normalizes for text length.

Result: accuracy 0.969, so a bit worse than bag of words. The difference is mostly human recall (0.93 vs 0.98). Maybe because TF-IDF gives low weight to function words, which are actually useful for telling human and AI apart, and because it removes the length information.

The results are very high, but dev comes from the same AI models and genres as train, so it would probably be lower on text from a new AI model.

In [18]:
tfidf = TfidfVectorizer(token_pattern=r"(?u)\b\w+\b")
X_train_tfidf = tfidf.fit_transform(train_texts)
X_dev_tfidf = tfidf.transform(dev_texts)

model_tfidf = LogisticRegression(max_iter=2000)
model_tfidf.fit(X_train_tfidf, y_train)
y_pred_tfidf = model_tfidf.predict(X_dev_tfidf)

print('Accuracy:', round(accuracy_score(y_dev, y_pred_tfidf), 3))
print('Confusion matrix:\n', confusion_matrix(y_dev, y_pred_tfidf))
print(classification_report(y_dev, y_pred_tfidf, target_names=['human', 'AI']))

Accuracy: 0.969
Confusion matrix:
 [[ 597   42]
 [  14 1142]]
              precision    recall  f1-score   support

       human       0.98      0.93      0.96       639
          AI       0.96      0.99      0.98      1156

    accuracy                           0.97      1795
   macro avg       0.97      0.96      0.97      1795
weighted avg       0.97      0.97      0.97      1795



## Bigram language models

#### We train one bigram model on human texts and one on AI texts. nltk.bigrams makes all word pairs and adds <s> and </s> at the start and end of each text. The model counts how often each word comes after another word.

To classify a dev text we compute its perplexity under both models. The model with the lowest perplexity is the least surprised by the text, so that is the guess. We use Laplace / add-k smoothing so bigrams we never saw in train don't get probability 0, and we add log probabilities instead of multiplying to avoid underflow.

Result: [fill in when you have run it]

In [19]:
def build_model(token_lists):
    model = defaultdict(Counter)
    for toks in token_lists:
        for w1, w2 in bigrams(toks, pad_left=True, pad_right=True,
                              left_pad_symbol='<s>', right_pad_symbol='</s>'):
            model[w1][w2] += 1
    totals = {w1: sum(nexts.values()) for w1, nexts in model.items()}
    return model, totals

human_model, human_totals = build_model([r['tokens'] for r in train if r['label'] == 0])
ai_model, ai_totals = build_model([r['tokens'] for r in train if r['label'] == 1])